In [ ]:
#MACHINE LEARNING MODEL FITTING
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline


In [ ]:
np.random.seed(0)
x=np.sort(np.random.rand(30))
y=np.sin(2*np.pi*x)+np.random.randn(30)*0.2
x=x.reshape(-1,1)

In [ ]:
plt.scatter(x,y, color='black')
plt.title("Original Data")
plt.show()


In [ ]:
#underfitting model
model1=make_pipeline(PolynomialFeatures(1), LinearRegression())
model1.fit(x,y)
x_test=np.linspace(0,1,100).reshape(-1,1)
y_pred1=model1.predict(x_test)
plt.scatter(x,y, color='black')
plt.plot(x_test,y_pred1)
plt.title("underfitting(High Bias)")
plt.show()

In [ ]:
model2=make_pipeline(PolynomialFeatures(4), LinearRegression())
model2.fit(x,y)
y_pred2=model2.predict(x_test)
plt.scatter(x,y, color='black')
plt.plot(x_test,y_pred2)
plt.title("Good fit (Balanced)")
plt.show()

In [ ]:
#OverFitting model(HighVariance)
model3=make_pipeline(PolynomialFeatures(15), LinearRegression())
model3.fit(x,y)
y_pred3=model3.predict(x_test)
plt.scatter(x,y, color='black')
plt.plot(x_test,y_pred3)
plt.title("Over fitting (High Variance)")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
df=pd.read_csv('D:\csv files\Banks_data.csv')


In [ ]:
df.head()

In [ ]:
target_col="bank_profit"
x=df.drop(columns=[target_col])
y=df[target_col]
print("x shape:", x.shape)
print("y shape:", y.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
x_scale=scaler.fit_transform(x)
x_scale=pd.DataFrame(x_scale,columns=x.columns)
print("x_scale shape:",x_scale.shape)

In [ ]:
from sklearn.model_selection import train_test_split, ShuffleSplit
x_train, x_test, y_train, y_test=train_test_split(x_scale,y,test_size=0.30, random_state=42)
print("Train size:",x_train.shape[0])
print("Test size:", x_test.shape[0])

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
lr=LinearRegression()
lr.fit(x_train,y_train)
y_train_pred_lr=lr.predict(x_train)
y_test_pred_lr=lr.predict(x_test)
lr_train_r2=r2_score(y_train,y_train_pred_lr)
lr_test_r2=r2_score(y_test,y_test_pred_lr)
print("Linear Regression (single split)")
print(f"Train r2: {lr_train_r2:.4f}")
print(f"Test r2: {lr_test_r2:.4f}")

In [ ]:
#Apply shuffle split and cross validation
from sklearn.model_selection import ShuffleSplit, cross_validate
shuffle_split=ShuffleSplit(n_splits=500, test_size=0.2,random_state=42)
cv_scores=cross_validate(lr,x_scale,y,cv=shuffle_split, scoring='r2', return_train_score=True)
train_scores=cv_scores['train_score']
test_scores=cv_scores['test_score']
print("corss validation:train score:",np.round(train_scores.mean(),2))
print("corss validation:test score:",np.round(train_scores.mean(),2))

In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.model_selection import ShuffleSplit
import numpy as np
shuffle_split_Lassocv=ShuffleSplit(n_splits=100,
                                   test_size=0.2,
                                   random_state=42)
lassocv_model=LassoCV(cv=shuffle_split_Lassocv,
                      random_state=42)
lassocv_model.fit(x_scale,y)
best_alpha_lassocv=lassocv_model.alpha_
mse_path=lassocv_model.mse_path_
mean_mse=mse_path.mean(axis=1)
best_mse=mean_mse[lassocv_model.alphas_==best_alpha_lassocv][0]
print("\n LassoCV Results:")
print("Best alpha found:",best_alpha_lassocv)
print("mean squared error for best alpha:",best_mse)

In [ ]:
from sklearn.linear_model import Lasso
final_lasso_model=Lasso(alpha=best_alpha_lassocv)
final_lasso_model.fit(x_scale,y)
lasso_coefficient=final_lasso_model.coef_
zero_coefficients=np.sum(lasso_coefficient==0)
non_zero_coefficients=np.sum(lasso_coefficient!=0)
print("\n Analysis of Lasso Coefficients(with best alpha):")
print("Number of zero coefficints:",zero_coefficients)
print("Number of non zero coefficients:",non_zero_coefficients)


In [ ]:
#get the indices of nonzero coefficients
non_zero_indices=np.nonzero(lasso_coefficient)[0]
selected_feature_names=x.columns[non_zero_indices]
print("\n Names of variables with non zero coefficients:")
print(selected_feature_names)
x_final=x_scale[selected_feature_names]
x_final.head()

In [ ]:
from sklearn.model_selection import ShuffleSplit, cross_validate
shuffle_split=ShuffleSplit(n_splits=500,
                           test_size=0.2,
                           random_state=42)
cv_scores=cross_validate(lr,x_final,y,
                         cv=shuffle_split,
                         scoring='r2',
                         return_train_score=True)
train_scores=cv_scores['train_score']
test_scores=cv_scores['test_score']
print("cross validatioin: training score:", np.round(train_scores.mean(),2))
print("cross validation: testing score:", np.round(test_scores.mean(),2))
                         

In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.model_selection import ShuffleSplit
cv=ShuffleSplit(n_splits=100,
                test_size=0.2,
                random_state=42)
lasso_cv=LassoCV(cv=cv,
                 random_state=42)
lasso_cv.fit(x_scale,y)
best_alpha_l1=lasso_cv.alpha_
print("best alpha(L1):",best_alpha_l1)


In [ ]:
from sklearn.linear_model import Lasso
lasso_model=Lasso(alpha=best_alpha_l1)
lasso_model.fit(x_scale,y)
coeff_l1=lasso_model.coef_
print((coeff_l1!=0).sum())
print((coeff_l1==0).sum())

In [ ]:
from sklearn.linear_model import RidgeCV
import numpy as np
ridge_cv=RidgeCV(alphas=np.logspace(-3,3,100),
                 cv=cv)
ridge_cv.fit(x_scale,y)
best_alpha_l2=ridge_cv.alpha_
print("Best Alpha L2:",best_alpha_l2)

In [ ]:
ridge_model=RidgeCV(alphas=best_alpha_l2)
ridge_model.fit(x_scale,y)
coeff_l2=ridge_model.coef_
print("Ridge coefficient:",coeff_l2)


In [ ]:
#ElasticNet (L1+L2)
from sklearn.linear_model import ElasticNetCV
elastic_cv=ElasticNetCV(l1_ratio=(0.1,0.5,0.7,0.9),
                        cv=cv,
                        random_state=42)
elastic_cv.fit(x_scale,y)
best_alpha_en=elastic_cv.alpha_
best_l1_ratio=elastic_cv.l1_ratio_
print("Best Alpha ratio",best_l1_ratio)

In [ ]:
l2_ratio=1-best_l1_ratio
print("L2 Ratio:",l2_ratio)

In [ ]:
#train Final ElasticNetCV
from sklearn.linear_model import ElasticNet
elastic_model=ElasticNet(alpha=best_alpha_en,
                         l1_ratio=best_l1_ratio)
elastic_model.fit(x_scale,y)
coeff_en=elastic_model.coef_
print("ElasticNet coefficients:",coeff_en)
print("zero coefficients:",(coeff_en==0).sum())
print("non zero coefficients:",(coeff_en!=0).sum())